# Two more datasets for the benchmark

A method comparison measured on one dataset risks describing that dataset
rather than the methods. Before benchmarking SHAP against LIME, this notebook
adds two more binary text classification tasks chosen to differ from IMDB in
domain and in task type:

- **Yelp review polarity**: sentiment again, but on business reviews, shorter
  and less essayistic than IMDB's movie reviews.
- **AG News, Business versus Sci/Tech**: a topic task, where the signal lives
  in entities and terminology rather than in evaluative language.

Both go through exactly the pipeline notebook 01 used for IMDB, now packaged
as `textfaith.training.build_checkpoint`: the same frozen e5-small-v2 encoder,
the same sampling protocol, the same MLP head, the same seeds. Keeping the
pipeline constant is the point, any faithfulness difference measured later is
then attributable to the data and the attribution method, not to pipeline
drift.

One deliberate addition over notebook 01: texts under 20 words are dropped.
The benchmark cuts up to 80 percent of a text in chunks of 8 words, and a
fifteen word stub leaves nothing measurable at that depth. IMDB is unaffected
in practice, its sampled reviews start at 37 words.

## Configuration

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIRECTORY = PROJECT_ROOT / "data"
ARTIFACTS_DIRECTORY = PROJECT_ROOT / "artifacts"

DATASET_ARCHIVES = {
    "yelp": "https://s3.amazonaws.com/fast-ai-nlp/yelp_review_polarity_csv.tgz",
    "agnews": "https://s3.amazonaws.com/fast-ai-nlp/ag_news_csv.tgz",
}

POOL_SIZE = 20000
RANDOM_SEED = 42

print(f"Data directory: {DATA_DIRECTORY}")
print(f"Pool size per dataset: {POOL_SIZE}")

Data directory: /home/user/projects/shapley/data
Pool size per dataset: 20000


## Download and parse

The fast.ai mirrors serve headerless CSVs with standard doubled quote
escaping, so plain `read_csv` parses them. What does need repair is inside the
text: the exports encode newlines as literal backslash n sequences and leave
stray backslashes where newlines once were. `clean_text` removes those along
with HTML tags, because a token like `dwindling\band` would otherwise become
a single word for the whitespace masker and quietly corrupt the attribution
measurements downstream.

Both raw corpora are far larger than needed, Yelp alone has 560,000 training
rows, and tokenizing them all just to sample 2000 would waste most of an hour.
A seeded stratified pool of 20,000 rows per dataset is drawn first; the
checkpoint sampling happens inside that pool.

In [2]:
import tarfile
import urllib.request

import pandas as pd
from sklearn.model_selection import train_test_split

from textfaith.training import clean_text

DATA_DIRECTORY.mkdir(exist_ok=True)

for name, url in DATASET_ARCHIVES.items():
    archive_path = DATA_DIRECTORY / url.rsplit("/", 1)[-1]
    extracted_directory = DATA_DIRECTORY / archive_path.name.removesuffix(".tgz")
    if not extracted_directory.exists():
        if not archive_path.exists():
            print(f"Downloading {url}")
            urllib.request.urlretrieve(url, archive_path)
        with tarfile.open(archive_path) as archive:
            archive.extractall(DATA_DIRECTORY, filter="data")
    print(f"{name}: {extracted_directory} ready")

agnews_raw = pd.read_csv(
    DATA_DIRECTORY / "ag_news_csv" / "train.csv",
    header=None,
    names=["label", "title", "description"],
)
agnews_binary = agnews_raw[agnews_raw["label"].isin([3, 4])].copy()
agnews_binary["text"] = (
    agnews_binary["title"] + " " + agnews_binary["description"]
).map(clean_text)
agnews_binary["label"] = (agnews_binary["label"] == 4).astype(int)

yelp_raw = pd.read_csv(
    DATA_DIRECTORY / "yelp_review_polarity_csv" / "train.csv",
    header=None,
    names=["label", "text"],
)
yelp_raw["text"] = yelp_raw["text"].map(clean_text)
yelp_raw["label"] = (yelp_raw["label"] == 2).astype(int)

pools = {}
for name, frame in [("yelp", yelp_raw), ("agnews", agnews_binary)]:
    pools[name], _ = train_test_split(
        frame[["text", "label"]],
        train_size=POOL_SIZE,
        random_state=RANDOM_SEED,
        stratify=frame["label"],
    )
    balance = pools[name]["label"].value_counts().to_dict()
    median_words = int(pools[name]["text"].str.split().str.len().median())
    print(f"{name}: pool of {len(pools[name])} from {len(frame)}, "
          f"balance {balance}, median words {median_words}")

print()
print("agnews sample:", pools["agnews"]["text"].iloc[0][:200])
print()
print("yelp sample:", pools["yelp"]["text"].iloc[0][:200])

/home/user/projects/shapley/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


yelp: /home/user/projects/shapley/data/yelp_review_polarity_csv ready


agnews: /home/user/projects/shapley/data/ag_news_csv ready


yelp: pool of 20000 from 560000, balance {1: 10000, 0: 10000}, median words 97
agnews: pool of 20000 from 60000, balance {1: 10000, 0: 10000}, median words 37

agnews sample: IBM, Borland upgrade developer tools OCTOBER 13, 2004 (COMPUTERWORLD) - Borland Software Corp. and IBM announced upgrades to their development tools this week designed to support heterogeneous environ

yelp sample: Don't miss the Steel City Sliders!


## Train the checkpoints

`build_checkpoint` reproduces notebook 01 end to end: deduplicate, drop texts
over 512 tokens or under 20 words, stratified sample of 2000, stratified 80/20
split, embed with the frozen encoder, fit
`MLPClassifier(hidden_layer_sizes=(128,), early_stopping=True)`. It is
idempotent per directory, an existing checkpoint is returned untouched, the
same cache discipline the attribution caches use.

In [3]:
from textfaith.training import build_checkpoint

LABEL_NAMES = {
    "yelp": {0: "negative", 1: "positive"},
    "agnews": {0: "business", 1: "sci/tech"},
}

metadata_by_dataset = {}
for name in ["yelp", "agnews"]:
    metadata_by_dataset[name] = build_checkpoint(
        pools[name],
        ARTIFACTS_DIRECTORY / f"{name}_2000",
        dataset_name=name,
        label_names=LABEL_NAMES[name],
        random_seed=RANDOM_SEED,
    )
    print(f"{name}: test accuracy {metadata_by_dataset[name]['test_accuracy']:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15840.78it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (943 > 512). Running this sequence through the model will result in indexing errors


yelp: test accuracy 0.9650


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20295.35it/s]

agnews: test accuracy 0.9075


## Round trip through the predictor

The benchmark will consume these checkpoints through the unmodified
`load_predictor`, so that exact path is what gets verified: load each new
checkpoint, re-score its test split, and require the accuracy to reproduce the
one recorded at training time.

In [4]:
import json

import numpy as np

from textfaith.checkpoint import load_predictor

summary_rows = []
for name in ["yelp", "agnews", "imdb"]:
    directory = ARTIFACTS_DIRECTORY / ("sample_2000" if name == "imdb" else f"{name}_2000")
    predict_probabilities, _ = load_predictor(directory)
    test_reviews = pd.read_parquet(directory / "test.parquet")
    predictions = predict_probabilities(test_reviews["text"].tolist()).argmax(axis=1)
    accuracy = float((predictions == test_reviews["label"].to_numpy()).mean())
    metadata = json.loads((directory / "metadata.json").read_text())
    assert np.isclose(accuracy, metadata["test_accuracy"], atol=1e-9), (
        f"{name}: loaded predictor accuracy {accuracy} does not reproduce "
        f"the recorded {metadata['test_accuracy']}"
    )
    summary_rows.append(
        {
            "dataset": name,
            "task": {
                "imdb": "movie sentiment",
                "yelp": "business sentiment",
                "agnews": "news topic",
            }[name],
            "train_size": metadata["train_size"],
            "test_size": metadata["test_size"],
            "test_accuracy": metadata["test_accuracy"],
            "median_test_words": int(
                test_reviews["text"].str.split().str.len().median()
            ),
        }
    )

summary = pd.DataFrame(summary_rows).set_index("dataset")
print("Every checkpoint reproduces its recorded accuracy through load_predictor")
summary

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19913.79it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16077.56it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18412.30it/s]

Every checkpoint reproduces its recorded accuracy through load_predictor


,task,train_size,test_size,test_accuracy,median_test_words
dataset,,,,,
yelp,business sentiment,1600,400,0.9650,90
agnews,news topic,1600,400,0.9075,38
imdb,movie sentiment,1600,400,0.9150,165
